# PageIndex batch QA

直接使用此仓库的原生 PDF ingest 和树导航问答，使用与 SuperIndex 相同的 golden question 和环境变量。`expected` 只写入结果，不进入模型提示词。默认 Flash；`PAGEINDEX_MODE=standard` 可比较经典建树。先在 Databricks Git folder 中打开此 notebook，然后按顺序运行。
需要 Python ≥3.10。setup 格先定位仓库，再从根目录安装依赖；Databricks 重启 Python 后从模型配置格继续。密钥可用 `.env` 或 Databricks secrets 注入。


本地已安装依赖时，可从仓库根目录自动执行并跳过 Databricks 安装格：
```bash
.venv/bin/python -m nbconvert --to notebook --execute notebooks/batch_qa.ipynb --TagRemovePreprocessor.enabled=True --TagRemovePreprocessor.remove_cell_tags databricks-setup --output-dir data/executed
```


In [ ]:
from pathlib import Path
_start = Path.cwd().resolve()
_setup_root = next(p for p in (_start, *_start.parents) if (p / "pageindex/client.py").is_file())


In [ ]:
%pip install -e "{_setup_root}" pandas openpyxl python-dotenv pycryptodome


In [ ]:
import os
if os.environ.get("DATABRICKS_RUNTIME_VERSION") and "dbutils" in globals():
    dbutils.library.restartPython()

In [ ]:
import os
# Databricks secrets can supply values here. Never put plaintext credentials in this notebook.
LLM_CONFIG: dict[str, str] = {
    # "OPENAI_MODEL": "your-model",
    # "OPENAI_BASE_URL": "https://gateway.example/v1",
    # "OPENAI_API_KEY": dbutils.secrets.get(scope="scope", key="key"),
    # "SUPERINDEX_REASONING_EFFORT": "",  # Explicit empty value disables reasoning_effort.
}
LLM_EXTRA_HEADERS: dict[str, str] = {}
for key, value in LLM_CONFIG.items():
    if value or key.endswith("REASONING_EFFORT"):
        os.environ[key] = value
os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")
os.environ.setdefault("OPENAI_AGENTS_DISABLE_TRACING", "1")

In [ ]:
import json
import shutil
import sys
import tempfile
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd().resolve()
REPO_DIR = next((p for p in (cwd, *cwd.parents) if (p / "pageindex" / "client.py").is_file() and (p / "pyproject.toml").is_file()), None)
if REPO_DIR is None:
    raise RuntimeError("Open this notebook inside the cloned PageIndex repository.")
ROOT_DIR = Path(os.environ.get("ROOT_DIR") or REPO_DIR).expanduser().resolve()
for env_file in dict.fromkeys([ROOT_DIR / ".env", cwd / ".env", REPO_DIR.parent / "SuperIndex" / ".env"]):
    if env_file.is_file():
        load_dotenv(env_file, override=False)
        print(f"Loaded .env: {env_file} (existing process variables take precedence)")
sys.path[:0] = [str(REPO_DIR), str(REPO_DIR / "notebooks")]
from batch_qa_support import ingest, load_questions, run_batch, sync_directory
from pageindex import PageIndexClient
import pageindex
print(f"Source: {Path(pageindex.__file__).resolve()}")


def env(name, default=None, cast=str):
    raw = os.environ.get(name)
    return default if raw is None or not raw.strip() else cast(raw.strip())


def shared(name, default=None, preserve_empty=False):
    for prefix in ("SUPERINDEX_", "PAGEINDEX_"):
        key = prefix + name
        if key in os.environ and (preserve_empty or os.environ[key].strip()):
            return os.environ[key].strip()
    return default


def under_root(value):
    p = Path(value).expanduser()
    return (p if p.is_absolute() else ROOT_DIR / p).resolve()


def bool_value(value):
    return str(value).lower() in ("1", "true", "yes", "on")


openai_model = env("OPENAI_MODEL")
openai_model = (openai_model if openai_model.startswith("openai/") else "openai/" + openai_model) if openai_model else None
CHAT_MODEL = shared("CHAT_MODEL", openai_model)
INDEX_MODEL = shared("INDEX_MODEL", openai_model)
if not CHAT_MODEL or not INDEX_MODEL:
    raise ValueError("Set SUPERINDEX_CHAT_MODEL and SUPERINDEX_INDEX_MODEL (legacy PAGEINDEX_* accepted), or OPENAI_MODEL to configure both. No implicit model fallback is used.")
BASE_URL = shared("BASE_URL", env("OPENAI_BASE_URL") if openai_model else None)
API_KEY = shared("API_KEY_OVERRIDE", env("OPENAI_API_KEY") if openai_model else None)
REASONING_EFFORT = shared("REASONING_EFFORT", preserve_empty=True) or None
chat_backend = {k: v for k, v in {"base_url": BASE_URL, "api_key": API_KEY}.items() if v}
index_backend = {k: v for k, v in {"api_base": BASE_URL, "api_key": API_KEY}.items() if v}
import subprocess
SOURCE_REVISION = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()


In [ ]:
# All input paths use the same env names and defaults as SuperIndex.
DATASET_PATH = under_root(env("DATASET_PATH", "data/questions.jsonl"))
if not os.environ.get("DATASET_PATH") and not DATASET_PATH.is_file():
    sibling = REPO_DIR.parent / "SuperIndex" / "data/questions.jsonl"
    DATASET_PATH = sibling if sibling.is_file() else REPO_DIR / "scripts/questions.json"
PDF_DIR = under_root(env("PDF_DIR", "data/pdfs"))
if not os.environ.get("PDF_DIR") and not PDF_DIR.is_dir():
    sibling = REPO_DIR.parent / "SuperIndex" / "data/pdfs"
    if sibling.is_dir():
        PDF_DIR = sibling
FIELD_MAP = env("FIELD_MAP", {"id": "id", "question": "question", "expected": "expected", "doc": "doc", "pages": None}, json.loads)
RECORDS_KEY = env("RECORDS_KEY")
LIMIT = env("LIMIT", 0, int)
TIMEOUT = env("TIMEOUT", 600.0, float)
MAX_TURNS = env("MAX_TURNS", 25)
MAX_TURNS = None if str(MAX_TURNS).lower() in ("none", "default") or int(MAX_TURNS) <= 0 else int(MAX_TURNS)
CONCURRENCY = env("CONCURRENCY", 1, int)
RESUME = env("RESUME", False, bool_value)
RATE_LIMIT_WAIT = env("RATE_LIMIT_WAIT", 60.0, float)
RATE_LIMIT_RETRIES = env("RATE_LIMIT_RETRIES", 5, int)
MODE = env("PAGEINDEX_MODE", "flash")
if MODE not in ("flash", "standard"):
    raise ValueError("PAGEINDEX_MODE must be flash or standard")
IS_DATABRICKS = bool(os.environ.get("DATABRICKS_RUNTIME_VERSION"))
# Namespaced outputs prevent reuse of SuperIndex's incompatible store.
configured_work = env("NB_WORK_DIR")
default_work = Path(tempfile.gettempdir()) / "pageindex_batch" if IS_DATABRICKS else "data/notebook"
base_work = under_root(configured_work or default_work)
WORK_DIR = base_work / "pageindex" / DATASET_PATH.stem
PERSIST_DIR = under_root(env("PERSIST_DIR")) / "pageindex" / DATASET_PATH.stem if env("PERSIST_DIR") else None
for output in (WORK_DIR, PERSIST_DIR):
    if output is not None and (output == PDF_DIR or output.is_relative_to(PDF_DIR) or PDF_DIR.is_relative_to(output)):
        raise ValueError(f"Output path overlaps read-only PDF input directory: {output}")
print(f"Work directory: {WORK_DIR} (NB_WORK_DIR={configured_work!r}); persist directory: {PERSIST_DIR}")
if PERSIST_DIR and PERSIST_DIR.exists() and not WORK_DIR.exists():
    shutil.copytree(PERSIST_DIR, WORK_DIR)
WORK_DIR.mkdir(parents=True, exist_ok=True)
STORE = WORK_DIR / "store"
RUNS_DIR = WORK_DIR / "runs"
RUNS_DIR.mkdir(exist_ok=True)
existing = sorted(p for p in RUNS_DIR.iterdir() if p.is_dir())
run_dir = existing[-1] if RESUME and existing else RUNS_DIR / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
run_dir.mkdir(exist_ok=True)
print(f"Dataset: {DATASET_PATH} (exists={DATASET_PATH.is_file()})")
print(f"PDF directory: {PDF_DIR} (exists={PDF_DIR.is_dir()})")
print(f"Mode: {MODE}; chat model: {CHAT_MODEL}; index model: {INDEX_MODEL}")
print(f"Work directory: {WORK_DIR}; persist directory: {PERSIST_DIR}")

In [ ]:
HKD_PER_USD = {2010: 7.7687, 2011: 7.7841, 2012: 7.7569, 2013: 7.7565, 2014: 7.7545, 2015: 7.7524, 2016: 7.7620, 2017: 7.7926,
               2018: 7.8376, 2019: 7.8351, 2020: 7.7559, 2021: 7.7727, 2022: 7.8306, 2023: 7.8292, 2024: 7.8030, 2025: 7.7956}
_hkd_table = ", ".join(f"{y}: {r:.4f}" for y, r in sorted(HKD_PER_USD.items()))

# --- 回答规范：追加在系统提示词（resolve_instructions()：SUPERINDEX_INSTRUCTIONS[_FILE] 或项目默认）之后，每题都生效 ---
ANSWER_RULES = f"""\
ANSWER RULES (these override any conflicting wording above):
- Language: always answer in English, whatever language the question is written in and whichever country or region it concerns. Keep company names, report titles and quoted document text as they appear; write everything else in English.
- Exchange rates: where a figure or a growth rate is reported on both an actual exchange rate (AER) basis and a constant exchange rate (CER) basis, use the AER figure and label it "AER". Never report CER figures, not even as a secondary number, unless the question explicitly asks for CER.
- Percentages: every percentage you calculate (growth, change, margin, share, ratio) is rounded to exactly two decimal places, e.g. 12.35%, 7.00%; show the formula and round the final calculated value. A percentage quoted verbatim from a document keeps the document's precision.
- Currency: state monetary amounts in the currency of the country or region the company belongs to (e.g. RMB for mainland China, SGD for Singapore, MYR for Malaysia, THB for Thailand, JPY for Japan), with one exception: for Hong Kong use US dollars (US$). If the document reports in a different currency, convert using the stated formula at the exchange rate the document states for that reporting period, and show the original figure and the rate used; if the documents state no rate, give the figure as reported and say that no conversion rate is available.
- Hong Kong dollars: convert every HK$ amount (including per-share figures in HK cents) to US$ using the stated formula. Use the HK$/US$ rate the report itself states for the reporting period when there is one (average rate for income and cash-flow items, closing rate for balance-sheet items); otherwise use the annual average rate of the reporting year from this table (HK$ per US$1, Federal Reserve H.10 annual averages): {_hkd_table}. For a year not in the table use the linked rate 7.80. Always show the original HK$ figure and the rate used.
- Format: open with the direct answer in one sentence (figure, unit, currency and reporting period), then the supporting figures, the calculation formula if any, and the source document and page.
"""

# Preserve the shared custom system instructions; PageIndex supplies its own native tool instructions.
custom_instructions = shared("INSTRUCTIONS", "")
instructions_file = shared("INSTRUCTIONS_FILE")
if instructions_file:
    custom_instructions = under_root(instructions_file).read_text(encoding="utf-8")
client = PageIndexClient(
    mode="local", storage_path=str(STORE), index_model=INDEX_MODEL,
    summary_model=INDEX_MODEL, chat_model=CHAT_MODEL,
    index_backend=index_backend, chat_backend=chat_backend,
    instructions=custom_instructions + "\n\n" + ANSWER_RULES,
)
client.instructions = (client.instructions or "") + "\n" + client.citation_prompt()
questions = load_questions(DATASET_PATH, FIELD_MAP, RECORDS_KEY)
if LIMIT > 0:
    questions = questions[:LIMIT]
print(f"Loaded {len(questions)} golden questions; expected answers stay out of model inputs.")

## 1. PDF ingest

原始 PDF 只读。每份文件独立记录失败；内容 SHA256、相对路径、模型及模式一致且 store 就绪时复用。Flash 仍需 LLM 摘要，不是离线 Markdown ingest。

In [ ]:
INDEX_SIGNATURE = {"model": INDEX_MODEL, "base_url": BASE_URL, "mode": MODE, "upstream_revision": SOURCE_REVISION}
documents = ingest(client, PDF_DIR, WORK_DIR, MODE, INDEX_SIGNATURE,
                   force=env("FORCE_INGEST", False, bool_value),
                   retries=RATE_LIMIT_RETRIES, wait=RATE_LIMIT_WAIT)
sync_directory(WORK_DIR, PERSIST_DIR)
print(f"Ready documents: {len(documents)}; report: {WORK_DIR / 'ingest_report.json'}")

## 2. 原生 retrieve + AI answer

每题沿章节树调用 PageIndex 自带工具并生成答案，不附加 BM25。输出完整工具参数、结果、读取页与引用；指定文档没有唯一匹配时失败，不退回全库。每题截止时间通过原生 async agent 取消执行。

In [ ]:
QA_SIGNATURE = {"upstream_revision": SOURCE_REVISION, "chat_model": CHAT_MODEL, "base_url": BASE_URL, "reasoning_effort": REASONING_EFFORT,
                "max_turns": MAX_TURNS, "timeout": TIMEOUT, "instructions": client.instructions,
                "extra_headers_signature": __import__("hashlib").sha256(json.dumps(LLM_EXTRA_HEADERS, sort_keys=True).encode()).hexdigest(), "field_map": FIELD_MAP}
records = run_batch(client, questions, documents, run_dir / "results.jsonl", QA_SIGNATURE,
                    concurrency=CONCURRENCY, resume=RESUME,
                    checkpoint=lambda: sync_directory(WORK_DIR, PERSIST_DIR),
                    retries=RATE_LIMIT_RETRIES, wait=RATE_LIMIT_WAIT,
                    timeout=TIMEOUT, max_turns=MAX_TURNS,
                    reasoning_effort=REASONING_EFFORT, extra_headers=LLM_EXTRA_HEADERS)
sync_directory(WORK_DIR, PERSIST_DIR)

## 3. 导出和检查

JSONL 保存完整检索轨迹；CSV/XLSX 与 SuperIndex 相同包含 question、expected、answer。

In [ ]:
import pandas as pd
from IPython.display import display
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

frame = pd.DataFrame([{"question": r["question"], "expected": r.get("expected") or "",
                       "answer": r["answer"] if not r["error"] else ""} for r in records],
                     columns=["question", "expected", "answer"])
frame.to_csv(run_dir / "answers.csv", index=False, encoding="utf-8-sig")
def clean_cell(value):
    return ILLEGAL_CHARACTERS_RE.sub("", value)[:32767] if isinstance(value, str) else value
clean = frame.map(clean_cell) if hasattr(frame, "map") else frame.applymap(clean_cell)
clean.to_excel(run_dir / "answers.xlsx", index=False)
sync_directory(WORK_DIR, PERSIST_DIR)
print(f"Success: {sum(r['status'] == 'success' for r in records)} / {len(records)}")
print(f"Results: {run_dir}")
for r in records:
    if r["error"]:
        print(f"{r['id']}: {r['error']}")
display(frame)